# YOLOv5 data validation on SoccerTrack v2 (match 117093)

Pre-SAM sanity check: convert GSR ground-truth boxes → a YOLOv5 dataset and train
YOLOv5 on Colab's GPU to confirm the data is sound and a detector can learn players.

**Logic lives in `src/`** (repo rule — CLAUDE.md §2). This notebook only orchestrates.

Order: setup → download ONE video → **verify box alignment** → build dataset → train → predict.

> Runtime → Change runtime type → **T4 GPU** before running.

## 1. Setup — clone repo + install deps

In [ ]:
import os, subprocess
REPO = 'https://github.com/wheredawoodat949/AI-Hackathon.git'
BRANCH = 'feat/pitch-eval-shaaz'  # branch carrying the YOLO converter
if not os.path.isdir('AI-Hackathon'):
    subprocess.run(['git','clone','--branch',BRANCH,REPO], check=True)
else:
    subprocess.run(['git','-C','AI-Hackathon','pull'], check=True)
%cd AI-Hackathon
!nvidia-smi -L

In [ ]:
# Data deps for download + converter (no torch needed yet; ultralytics pulls its own).
!pip -q install gdown PyYAML python-dotenv numpy opencv-python-headless pillow
# Make `import src...` work from the repo root.
import sys; sys.path.insert(0, os.getcwd())

## 2. Download annotations (gsr/bas/raw — small, no quota)

Pull the annotations only (`--no-videos`). These are small and not rate-limited.

**The ~3GB video is fetched separately in step 2b** because Google Drive throttles
large public files ("Too many users have downloaded this file recently"). We get the
video via your mounted Drive instead — see 2b.

In [ ]:
!python -m src.data.download --match 117093 --source drive --no-videos
!ls -lh data/gsr/117093/ data/raw/117093/

## 2b. Get the big files via your Google Drive (bypasses the download quota)

Google Drive rate-limits large public files ("Too many users have downloaded this
file recently"). Two files trip it: the **video** and the **2nd-half GSR JSON** (both
multi-GB). So we copy them from **your** Drive instead — no anonymous quota.

**One-time browser step — add a shortcut to EACH of these** (open the mirror:
https://drive.google.com/drive/folders/1N2Qx2qkFgRtpbHitl2Vh6sLVYGgqkWwn ,
right-click the file → **Organize → Add shortcut → My Drive**):
1. `videos/117093/` → **`117093_panorama_2nd_half.mp4`** (the NORMAL one, *not* calibrated)
2. `gsr/117093/` → **`117093_2nd.json`**

Then run the cell below: it mounts your Drive, finds each shortcut anywhere in MyDrive,
and copies it into the path the loader expects. Re-runs skip files already present.

In [ ]:
import os, glob, shutil

# Files we pull from YOUR Drive (add a shortcut to each in MyDrive first):
#   filename -> destination path the loader/pipeline expects.
# The big GSR JSON and the video both hit the public download quota, so we copy
# them from your mounted Drive instead of gdown.
DRIVE_FILES = {
    '117093_panorama_2nd_half.mp4': 'data/videos/117093/117093_panorama_2nd_half.mp4',
    '117093_2nd.json':              'data/gsr/117093/117093_2nd.json',
}

need = {name: dest for name, dest in DRIVE_FILES.items() if not os.path.exists(dest)}
if not need:
    print('all Drive files already present.')
else:
    from google.colab import drive
    drive.mount('/content/drive')   # click through the auth prompt
    for name, dest in need.items():
        # Shortcut may sit in MyDrive root or any subfolder — search for it.
        hits = (glob.glob(f'/content/drive/MyDrive/CalHacks_Video/**/{name}', recursive=True)
                + glob.glob(f'/content/drive/MyDrive/CalHacks_Video/{name}'))
        assert hits, (f'{name} not found in your Drive. Add a shortcut to it in '
                      'MyDrive (right-click -> Organize -> Add shortcut).')
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        print(f'copying {name} ({os.path.getsize(hits[0])/1e9:.2f} GB)...')
        shutil.copy(hits[0], dest)
        print('  ->', dest)

# Guard: confirm everything landed and the video is the NORMAL panorama.
for name, dest in DRIVE_FILES.items():
    assert os.path.exists(dest), f'missing after copy: {dest}'
assert not glob.glob('data/videos/117093/*calibrated*'), 'calibrated video present — remove it'
print('OK:', list(DRIVE_FILES.values()))

## 3. Sanity-check box alignment

GSR `bbox_image` boxes are in the **normal panorama's native 4096×1080 space** —
verified offline: box x-extent ≈ [0, 4096], y-extent ≈ [135, 977], and `raw/`
`mapx/mapy` target a 4096×1080 grid. So no rescale/remap is needed; `build(img_wh=None)`
(video-native size) is correct.

This cell draws boxes on real frames as a visual confirmation. **Expected:** boxes sit
on players. In any single frame players often cluster up-pitch (one end), so some boxes
near the top edge are normal — they're real players at the far touchline, not a bug.
Across frames the boxes track the play and reach the bottom of the frame too.

(The earlier misalignment was from the *calibrated* panorama / from rescaling boxes by
3840×1504 — both wrong. The normal panorama at native size is the right pairing.)

In [ ]:
import glob
from src.data.yolo_dataset import overlay_check

# IMPORTANT: pick the NORMAL panorama, not the calibrated one. 'panorama_2nd' is a
# substring of 'calibrated_panorama_2nd', so we must exclude 'calibrated' explicitly.
vids = [v for v in glob.glob('data/videos/117093/*panorama_2nd*') if 'calibrated' not in v]
assert vids, "normal panorama_2nd video not found — re-run the download cell (step 2)."
VIDEO = vids[0]
print('video:', VIDEO)   # expect 117093_panorama_2nd_half.mp4 (NOT calibrated_...)

pngs = overlay_check('data', '117093', VIDEO, half=2, image_ids=(1000, 20000, 50000))

from IPython.display import Image, display
for p in pngs:
    print(p); display(Image(filename=str(p)))

### 3b. (optional) confirm boxes span the full frame across time

Quick numeric check that boxes aren't stuck at the top: print the bbox y-extent over
many frames. You should see y reach well past the midline (~down to ~977 of 1080),
confirming the per-frame clustering in step 3 is just where players happened to be.
No remap is applied — boxes already live in video space.

In [ ]:
from src.data.loader import load_match
m = load_match('data', '117093')
ys = []
for fr in m.gsr_frames(2)[:30000:1000]:   # sample across the half
    for e in fr.entities:
        if e.bbox_image:
            ys.append(e.bbox_image[1] + e.bbox_image[3])  # box bottom y
print(f'box-bottom y over {len(ys)} boxes: min={min(ys)}, max={max(ys)} (frame height 1080)')
print('-> boxes span top-to-bottom of the frame; alignment is correct, no remap needed.')

## 4. Build the YOLOv5 dataset

`img_wh=None` uses the video's native 4096×1080 (the boxes' space — confirmed above).
`stride=25` ≈ 1 fps; `limit` caps frames for a quick run. Bump both for a fuller set.

In [ ]:
from src.data.yolo_dataset import build
build('data', '117093', VIDEO, out_dir='yolo_ds',
      half=2, stride=25, limit=400, img_wh=None)  # img_wh=None -> video native size
!cat yolo_ds/data.yaml && echo '---' && ls yolo_ds/images/train | head

## 5. Train YOLOv5 (Ultralytics) — with data augmentation

Ultralytics serves YOLOv5 weights (`yolov5su.pt`) through the same API. Augmentation
is built in — you pass it as `train()` args (no custom pipeline needed).

**Soccer-specific choices** (this is a fixed-camera panorama, so geometry matters):
- ✅ **Horizontal flip** (`fliplr`) — mirrored pitch is valid soccer.
- ✅ **Color jitter** (`hsv_*`) — handles lighting / turf / weather variation.
- ⚠️ **Slight rotation** (`degrees=5`) only — real pitches aren't steeply tilted.
- ❌ **Vertical flip** (`flipud=0`) — would put sky at the bottom; never seen at test.
- ⚠️ **Mosaic** shrinks already-tiny players; we disable it for the final epochs
  (`close_mosaic`) so training finishes on clean frames.

Want a clean before/after number for Arize? Run once with `aug={}` (defaults) and once
with this block, compare `model.val()` mAP.

In [ ]:
!pip -q install ultralytics
from ultralytics import YOLO
model = YOLO('yolov5su.pt')  # YOLOv5-small (Ultralytics build)

# --- Data augmentation (all built into Ultralytics; just pass as train args) -----
# Tuned for a FIXED-perspective panoramic pitch: the camera never moves, sky is
# always up / grass always down. So we keep augmentations that yield *valid* soccer
# scenes and disable the ones that don't.
aug = dict(
    # Color distortions — KEEP (lighting/weather/turf vary across matches).
    hsv_h=0.015,   # hue jitter
    hsv_s=0.7,     # saturation jitter
    hsv_v=0.4,     # brightness jitter

    # Horizontal flip — KEEP. A mirrored pitch (attack left<->right) is valid soccer.
    fliplr=0.5,
    # Vertical flip — OFF. Would put stands/sky at the bottom: a view never seen at
    # inference. Set >0 only if you deliberately want that (you don't here).
    flipud=0.0,

    # Slight rotation — small only. Real broadcast pitches are never steeply tilted.
    degrees=5.0,

    # Mild geometric jitter for robustness (players are tiny, so keep gentle).
    translate=0.1,
    scale=0.5,
    shear=0.0,
    perspective=0.0,

    # Mosaic stitches 4 imgs -> shrinks already-tiny players; turn it off for the
    # last 10 epochs so the model finishes on clean full-size frames.
    mosaic=1.0,
    close_mosaic=10,
    mixup=0.0,
)

model.train(data='yolo_ds/data.yaml', epochs=25, imgsz=1280, batch=16,
            project='runs', name='st_yolov5', exist_ok=True, **aug)

In [ ]:
from ultralytics import YOLO

# Load the state you just trained (full checkpoint, not best.pt)
model = YOLO('runs/detect/runs/st_yolov5/weights/last.pt')

model.train(
    data='yolo_ds/data.yaml',
    epochs=10,                 # 10 MORE epochs on top of the trained weights
    imgsz=1280, batch=16,
    project='runs', name='st_yolov5_more',   # new folder so you don't overwrite the first run
    exist_ok=True,
    **aug,                     # reuse your same augmentation dict
)

## 6. Validate — predict on a held-out frame

Run the trained model on a val image and view the detections. Boxes on players =
the data validates end-to-end and a detector learns it. ✅

In [ ]:
import glob
from IPython.display import Image, display

val_imgs = sorted(glob.glob('yolo_ds/images/val/*.jpg'))
res = model.predict(val_imgs[0], imgsz=1280, conf=0.25, save=True,
                    project='runs', name='pred', exist_ok=True)
print('detections:', len(res[0].boxes))

# res[0].save_dir is the real output folder (handles the detect/ subdir for you)
saved = sorted(glob.glob(f'{res[0].save_dir}/*.jpg'))
assert saved, f'no annotated image in {res[0].save_dir}'
display(Image(filename=saved[0]))

## Next
- If detections look right, the GSR data + bbox pipeline are validated for SAM/Phase 2.
- `metrics = model.val()` gives mAP for an Arize before/after story (see docs/ML_DIRECTIONS.md).
- The same boxes feed Role B's homography → minimap (raw/117093_homography.npy).